# Workshop 3 — Navigation: Reimplementing `go_to`

**Estimated time:** ~90 minutes  
**Prerequisites:** Workshops 1 & 2

The built-in `go_to` skill is a black box. In this workshop you will build it from scratch in two ways:

- **Part A — Global-state navigation:** Use pose data from `scene_state` and `robot_status` to compute heading errors and drive with a proportional controller.
- **Part B — Vision-only navigation:** Use only the robot's camera (color detection + blob area) to navigate, without any privileged position data.

Understanding both approaches is crucial for real-world robotics where GPS/localization may not be available.

> **Workshop baseline assumption:** The examples below are intended to run in a controlled setup where the target is visible and the direct route is not blocked by obstacles. Handling blocked paths, target loss, active scanning, and obstacle-aware detours is introduced conceptually here and becomes project/stretch work.

> ⚠️ **Before you begin:** Close any open simulator windows from previous workshops. Each notebook creates a fresh robot — two open viewer tabs simultaneously causes issues.

## Section 1 — Quick Setup

In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv opencv-python matplotlib

In [ ]:
# Same setup as Workshop 1 — see that notebook for detailed explanation
import asyncio
import math
import os
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── Key loader: tries .env first, then Colab Secrets ────────────────────────
def _load_keys():
    # Mode B: local .env file
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # Mode A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")

In [ ]:
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop3-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> Open the viewer URL in **Google Chrome**, wait for the scene to load, then run the cell below.

In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """Poll until the viewer joins and skills become available."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")

---
# Part A — Global-State Navigation

In Part A we use world coordinates from `scene_state` and `robot_status` to navigate. This is the most reliable approach but requires access to privileged simulation data.

## Part A setup: clear path to `pad_C`

Before starting Section 2, use the Chrome viewer to move the robot to a spot where there is a mostly straight, unobstructed path from the robot to `pad_C`. The robot does **not** need to face `pad_C` yet; Section 3 will compute the heading error and turn the robot.

This setup keeps Part A focused on coordinate geometry and proportional control. If another asset blocks the straight route, the simple `drive_to_distance` baseline may get stuck because it does not plan around obstacles.


## Section 2 — Robot Pose and World Coordinates

The warehouse uses a 2D coordinate system:
- **yaw = 0°** → robot faces the +x direction
- **yaw = 90°** → robot faces the +y direction
- **bearing** = `atan2(dy, dx)` — world-frame angle to a target
- **heading_error** = how many degrees the robot needs to turn to face the target

In [ ]:
# Read robot pose and target position
status = await session.state.get("robot_status")
scene  = await session.state.get("scene_state")

rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
yaw    = status.robot.pose.yaw_deg

pad_c  = scene.entities["pad_C"]
tx, ty = pad_c.pose.position[0], pad_c.pose.position[1]

print(f"Robot   : x={rx:.2f}, y={ry:.2f}, yaw={yaw:.1f}°")
print(f"pad_C   : x={tx:.2f}, y={ty:.2f}")

# Geometry
dist          = math.hypot(tx - rx, ty - ry)
bearing       = math.degrees(math.atan2(ty - ry, tx - rx))
heading_error = (bearing - yaw + 180) % 360 - 180

print(f"\nDistance      : {dist:.2f} m")
print(f"Bearing       : {bearing:.1f}° (world frame)")
print(f"Heading error : {heading_error:.1f}° (how much to turn)")

## Section 3 — Turning to Face a Target

In [ ]:
async def turn_to_face(session, target_pos):
    """Turn the robot to face target_pos = [x, y, ...]."""
    state = await session.state.get("robot_status")
    rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
    yaw = state.robot.pose.yaw_deg
    tx, ty = target_pos[0], target_pos[1]

    bearing = math.degrees(math.atan2(ty - ry, tx - rx))
    heading_error = (bearing - yaw + 180) % 360 - 180
    print(f"  Heading error: {heading_error:+.1f}°")

    if abs(heading_error) < 5:
        print("  Already facing target.")
        return

    # wz > 0 turns left, wz < 0 turns right
    wz = 0.4 if heading_error > 0 else -0.4
    # duration = angle / angular_speed  (in radians)
    duration = min(abs(heading_error) / (abs(wz) * 57.296), 8.0)

    await session.invoke(
        "set_velocity",
        {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": duration},
        timeout_s=30,
    )
    print(f"  Turned {heading_error:+.0f}° in {duration:.2f}s")


# Demonstrate: turn to face pad_C
scene = await session.state.get("scene_state")
pad_c_pos = scene.entities["pad_C"].pose.position
print("Turning to face pad_C...")
await turn_to_face(session, pad_c_pos)

## Section 4 — Driving Toward a Target

In [ ]:
async def drive_to_distance(session, target_pos, tolerance=0.6):
    """Drive forward until within tolerance meters of target_pos."""
    for step in range(20):
        state = await session.state.get("robot_status")
        rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
        dist = math.hypot(target_pos[0] - rx, target_pos[1] - ry)
        print(f"  Step {step+1}: dist={dist:.2f}m")
        if dist <= tolerance:
            print("  Arrived!")
            return True
        vx = min(0.8, dist * 0.4)  # proportional speed
        await session.invoke(
            "set_velocity",
            {"vx": vx, "vy": 0.0, "wz": 0.0, "duration_s": 0.8},
            timeout_s=30,
        )
    print("  Max iterations reached.")
    return False


# Demonstrate: drive toward pad_C after facing it
print("Driving toward pad_C...")
await drive_to_distance(session, pad_c_pos, tolerance=0.8)

> **Baseline limitation:** `drive_to_distance` drives toward a target coordinate but does not plan around other assets. If something blocks the straight route, the robot may get stuck or stop far from the target. For this workshop, run the required examples in a setup where the route is clear.

## Section 5 — Full `my_go_to_global`

Combining turn and drive in an iterative correction loop:

In [ ]:
async def my_go_to_global(session, entity_id, tolerance=0.6, max_iters=10):
    """Navigate to entity_id using scene_state + robot_status."""
    scene = await session.state.get("scene_state")
    entity = scene.entities.get(entity_id)
    if entity is None:
        print(f"Entity '{entity_id}' not found in scene.")
        return False
    target_pos = entity.pose.position

    for i in range(max_iters):
        state = await session.state.get("robot_status")
        rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
        dist = math.hypot(target_pos[0] - rx, target_pos[1] - ry)
        print(f"Iter {i+1}: dist={dist:.2f}m to {entity_id}")
        if dist <= tolerance:
            print(f"Reached {entity_id}!")
            return True
        await turn_to_face(session, target_pos)
        await drive_to_distance(session, target_pos, tolerance=tolerance)

    print("Max iterations reached.")
    return False


# Demo: navigate to pad_C
print("my_go_to_global → pad_C")
success = await my_go_to_global(session, "pad_C")
print(f"Result: {success}")

---
# Part B — Vision-Only Navigation

In the real world, a robot may not have access to precise pose data. In Part B we navigate using only the robot's camera — no `scene_state`, no `robot_status`.

## Section 6 — Why Vision-Only Matters

- **Sim-to-real transfer:** The gap between simulation and physical hardware means privileged data (exact coordinates) rarely exists on real robots.
- **Universal sensors:** A camera is the most common onboard sensor; global information is not always available on a robot
- **Challenge:** Vision is noisy, lighting varies, objects can be occluded — requiring more robust algorithms.

The key insight: instead of asking "where is the cube in world coordinates?", we ask "where is the cube in my camera view, and how do I close the gap?"

## Section 7 — Self-Contained `perceive()` Function

This is the same function from Workshop 2, copied here so this notebook is self-contained:

In [ ]:
# Copied from Workshop 2 — self-contained, no import from another notebook needed
async def perceive(session):
    """Return {color: {angle_deg, blob_area}} for each visible cube color."""
    jpeg = await session.get_vision("pov")
    img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    COLOR_RANGES = {
        "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
                   (np.array([160,50, 50]), np.array([180, 255, 255]))],
        "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
        "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
        "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
    }

    result = {}
    for color, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, lo, hi)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        if area < 200:
            continue
        M = cv2.moments(largest)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        angle_deg = (cx - w / 2) / (w / 2) * 30.0
        result[color] = {"angle_deg": round(angle_deg, 1), "blob_area": int(area)}

    return result


# Quick test
obs = await perceive(session)
print("Current perception:")
for color, data in obs.items():
    print(f"  {color}: angle={data['angle_deg']:+.1f}°, area={data['blob_area']}px²")

## Section 8 — Centering on a Target Color

Turn the robot until the target color's blob is within ±10° of the camera center.

In [ ]:
async def center_on_color(session, target_color, angle_tolerance=10.0, max_steps=12):
    """Turn until target_color blob is within angle_tolerance of camera center."""
    for step in range(max_steps):
        obs = await perceive(session)
        if target_color not in obs:
            print(f"  Step {step+1}: {target_color} not visible — scanning...")
            await session.invoke(
                "set_velocity",
                {"vx": 0.25, "vy": 0.0, "wz": 0.3, "duration_s": 1.0},
                timeout_s=15,
            )
            continue

        angle = obs[target_color]["angle_deg"]
        print(f"  Step {step+1}: {target_color} at {angle:+.1f}°")

        if abs(angle) <= angle_tolerance:
            print("  Target centered!")
            return True

        # Turn toward center: positive angle = object is right → turn right (wz negative)
        wz = -0.3 if angle > 0 else 0.3
        await session.invoke(
            "set_velocity",
            {"vx": 0.25, "vy": 0.0, "wz": wz, "duration_s": 0.8},
            timeout_s=15,
        )

    print("  Max steps reached.")
    return False


print("Centering on red cube...")
success = await center_on_color(session, "red")
print(f"Centered: {success}")

## Section 9 setup: visible target and clear route

Before running `drive_toward_color`, use the Chrome viewer to move the robot to a spot where at least one cube is visible in the robot camera and the route to that cube is mostly clear. The robot does **not** need to be perfectly centered on the cube; Section 8 can center it first.

If an obstacle blocks the route or the target leaves the camera view, the baseline may fail. Treat that as a useful observation for the stretch/project discussion rather than something the required workshop code must solve.


## Section 9 — Driving by Blob Size

As the robot approaches an object, its blob in the image grows. We use blob area as a proximity proxy: drive forward until the area exceeds a threshold.

**Rule of thumb:** `ARRIVAL_AREA = 8000 px²` corresponds roughly to the robot being within arm's reach of the cube.

In [ ]:
ARRIVAL_AREA = 8000  # pixels² — tune if needed

async def drive_toward_color(session, target_color, arrival_area=ARRIVAL_AREA, max_steps=15):
    """Drive forward toward target_color using blob area as proximity proxy."""
    for step in range(max_steps):
        obs = await perceive(session)
        if target_color not in obs:
            print(f"  Step {step+1}: {target_color} not visible")
            return False

        area  = obs[target_color]["blob_area"]
        angle = obs[target_color]["angle_deg"]
        print(f"  Step {step+1}: area={area}px², angle={angle:+.1f}°")

        if area >= arrival_area:
            print("  Arrived (blob large enough)!")
            return True

        # Re-center every other step to stay on course
        if step % 2 == 1 and abs(angle) > 10:
            wz = -0.3 if angle > 0 else 0.3
            await session.invoke(
                "set_velocity",
                {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": 0.5},
                timeout_s=15,
            )
        else:
            await session.invoke(
                "set_velocity",
                {"vx": 0.5, "vy": 0.0, "wz": 0.0, "duration_s": 1.0},
                timeout_s=15,
            )

    print("  Max steps reached.")
    return False


print("Driving toward red cube...")
success = await drive_toward_color(session, "red")
print(f"Arrived: {success}")

## Section 10 — Full `my_go_to_visual`

In [ ]:
async def my_go_to_visual(session, target_color, arrival_area=ARRIVAL_AREA):
    """Navigate to a cube of target_color using only the camera. No scene_state used."""
    print(f"Navigating to {target_color} cube (vision only)...")
    centered = await center_on_color(session, target_color)
    if not centered:
        print("Could not center on target.")
        return False
    return await drive_toward_color(session, target_color, arrival_area=arrival_area)


# Demo on a visible cube color
obs = await perceive(session)
if obs:
    demo_color = next(iter(obs))  # pick first detected color
    print(f"Demo: navigating to {demo_color}")
    await my_go_to_visual(session, demo_color)
else:
    print("No cubes visible — move the robot closer to the conveyor belt first.")

---
## Further Questions: Obstacle-Aware Vision Navigation

The baseline functions above deliberately keep navigation simple. Think about how you would extend them when the direct route is blocked or the target leaves the camera view:

- How could `set_head` help the robot scan left/right without immediately turning its whole body?
- When should the robot step back before trying to re-center the target?
- How could the robot remember the last-seen target direction?
- What evidence would tell the robot that it is stuck behind an obstacle rather than simply moving slowly?

These questions are not required for the main workshop exercises. They connect directly to the advanced project level.

---
## Exercises

### Exercise 1 — Print Positions and Distance (Part A)

Read the robot's current `(x, y)` position and the position of `pad_C` from `scene_state`. Compute and print the straight-line distance between them.

In [ ]:
## Exercise 1: Print robot position, pad_C position, and distance
# YOUR CODE HERE

# Expected output:
# Robot: (1.23, 0.45)
# pad_C: (5.60, 2.10)
# Distance: 4.62 m

### Exercise 2 — Turn to Face pad_C (Part A)

Get the position of `pad_C` from `scene_state` and call `turn_to_face(session, pad_c_pos)`. Print the heading error before and after the turn.

In [ ]:
## Exercise 2: Turn to face pad_C
# YOUR CODE HERE

# Expected output:
# Before: heading_error = +42.3°
# [turn executes]
# After: heading_error ≈ 0–5°

### Exercise 3 — Drive to pad_C (Part A)

After turning to face `pad_C`, call `drive_to_distance(session, pad_c_pos, tolerance=0.8)`. Print the distance each iteration.

In [ ]:
## Exercise 3: Drive to pad_C
# YOUR CODE HERE

# Expected output:
# Step 1: dist=4.62m
# Step 2: dist=3.45m
# ...
# Arrived!

### Exercise 4 — Test `my_go_to_global` on pad_D (Part A)

Call `my_go_to_global(session, "pad_D")`. After it returns, verify arrival by printing the final distance to `pad_D` from `scene_state`.

In [ ]:
## Exercise 4: Navigate to pad_D and verify
# YOUR CODE HERE

# Expected: final distance < 0.6m
# e.g.: "Arrived at pad_D. Final distance: 0.42m"

### Exercise 5 (Stretch) — Compare Paths: `my_go_to_global` vs Built-in `go_to` (Part A)

Log the robot's position every 1 second during both a `my_go_to_global` run and a built-in `go_to` run to `pad_B`. Plot both paths on a 2D scatter plot using `matplotlib`.

In [ ]:
## Exercise 5 (stretch): Compare paths to pad_B
# YOUR CODE HERE

# Hint:
# 1. Start a background task that polls robot_status every 1s and appends to a list
# 2. Run my_go_to_global(session, "pad_B")
# 3. Repeat with built-in go_to
# 4. plt.plot(xs, ys) for each path, mark start/end, label them

### Exercise 6 — Measure Camera Angle Offset (Part B)

Call `perceive(session)` and print the `angle_deg` for the red cube. Then compute the ground truth angle using `scene_state` + `robot_status`. Compare the two values.

In [ ]:
## Exercise 6: Camera angle vs scene_state ground truth
# YOUR CODE HERE

# Expected output:
# Camera angle: +14.2°
# Scene truth:  +12.8°
# Error:         1.4°

### Exercise 7 — Center on Red Cube (Part B)

Call `center_on_color(session, "red")`. Print the angle offset each step. Observe how it converges toward 0°.

In [ ]:
## Exercise 7: Center on red cube
# YOUR CODE HERE

# Expected output:
# Step 1: red at +18.5°
# Step 2: red at +9.2°
# Step 3: red at +3.1° — Target centered!

### Exercise 8 — Drive Toward Red Cube (Part B)

After centering, call `drive_toward_color(session, "red")`. Print the blob area each step. Observe how it grows as the robot approaches.

In [ ]:
## Exercise 8: Drive toward red cube by blob size
# YOUR CODE HERE

# Expected output:
# Step 1: area=1240px², angle=+2.1°
# Step 2: area=2870px², angle=+1.8°
# ...
# Arrived (blob large enough)!

### Exercise 9 (Stretch) - Vision-Only Navigation and Failure Analysis (Part B)

Call `my_go_to_visual(session, "blue")` in a controlled setup where the blue cube is visible. After it completes, check the actual distance to the blue cube using `scene_state`.

Then try one harder case where the target is lost or the route is blocked. What fails first: centering, approach, target visibility, or obstacle handling?

In [ ]:
## Exercise 9 (stretch): Vision-only navigation + failure analysis
# YOUR CODE HERE

# Expected in a controlled setup:
# Navigation completes or gets close to the target.
# Final distance to blue cube: roughly 0.5?1.5m.
#
# Expected in an obstacle/lost-target setup:
# The baseline may fail. Record where it fails and what recovery behavior
# would be needed, such as set_head scanning, stepping back, or re-centering.

---
## Cleanup

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")